# FRM · Exp 1 — FRM vs Eccentricity (make-or-break, GPU)

Trains FRM out-of-fold (KL distillation) and asks: does the LEARNED module beat the geometric eccentricity baseline, **especially on far context**? Cheap ranking proxies run on all samples; the decisive answer-preservation metric runs the frozen VLM on a balanced far/near subset (needs GPU).

In [ ]:
# 1) mount Google Drive (dataset + outputs live here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) clone the repo containing frm/  (EDIT REPO_URL if your repo name differs)
import os
REPO_URL = "https://github.com/shubhamOjha1000/AAAI_2027_code.git"
REPO_DIR = "/content/AAAI_2027_code"
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull
# locate frm/ (it sits at the repo root)
FRM = os.path.join(REPO_DIR, "frm")
if not os.path.exists(os.path.join(FRM, 'config.py')):
    import subprocess
    hit = subprocess.check_output(['bash','-lc',
        f"find {REPO_DIR} -name config.py -path '*frm*' | head -1"]).decode().strip()
    FRM = os.path.dirname(hit)
assert os.path.exists(os.path.join(FRM, "config.py")), "frm/ not found — check REPO_URL"
%cd $FRM

In [ ]:
# 3) install deps
!pip -q install -r requirements.txt

In [ ]:
# 4) point at the dataset on Drive + output dir; put frm/ on the path
import os, sys
os.environ["DATA_DIR"] = "/content/drive/MyDrive/wearvqa_gaze_only"
os.environ["FRM_OUT_DIR"] = "/content/drive/MyDrive/frm_out"
sys.path.insert(0, os.getcwd())
import config as C; print('DATA_DIR =', C.DATA_DIR); print('OUT_DIR  =', C.OUT_DIR)

### Run (set preserve_limit lower for a quick pass)

In [ ]:
from experiments import exp1_frm_vs_eccentricity as e1
result = e1.run(preserve_limit=120)

### Verdict

In [ ]:
import json
print(json.dumps(result.get('preserve', {}), indent=2))
print(json.dumps(result['proxies'], indent=2))

**Success/kill:** FRM answer-kept > Eccentricity on the FAR subset → FRM justified. FRM ≈ Eccentricity → drop FRM, use the geometric baseline.